# 4. Pre/Post Energy Community Analysis

## Overview
This notebook separates the integrated dataset into two distinct groups for causal impact analysis:
- **Before**: Periods before energy community creation (baseline/control periods)
- **After**: Periods after energy community creation (treatment periods)

This split enables difference-in-differences analysis and impact assessment.

### Inputs
- `3_AllDataAgregation_output/month_con_full_AMP.csv` – Complete integrated dataset

### Outputs
- `4_SeparateData_output/data_before_1acc.csv` – Pre-community periods for baseline analysis
- `../input/data_before_1acc.csv` – Pre-community periods - Use in modeling
- `4_SeparateData_output/data_after_1acc.csv` – Post-community periods for treatment analysis
- `../input/data_after_1acc.csv` – Post-community periods - Use in modeling

### Processing Steps
1. Load the integrated consumption & environmental dataset
2. Split data based on energy community timeline (`months_since_acc_cer`)
3. Remove temporal classification columns from final output
4. Save balanced datasets ready for modeling

In [2]:

# Load Integrated Dataset

import pandas as pd
import os

output_dir = '../input'
os.makedirs(output_dir, exist_ok=True)
output_dir = '4_SeparateData_output'
os.makedirs(output_dir, exist_ok=True)

print("Loading integrated dataset...")
df = pd.read_csv("3_AllDataAgregation_output/month_con_full_AMP.csv")

print(f"Total records: {len(df):,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Parishes: {df['parish_code'].nunique()}")


# Split by Energy Community Timeline

print("\nSplitting data by community timeline...")

# Before: months before energy community creation (baseline)
df_before = df[df['months_since_acc_cer'] < 0].copy()

# After: months after or at energy community creation (treatment)
df_after = df[df['months_since_acc_cer'] >= 0].copy()

print(f"Before community: {len(df_before):,} records ({len(df_before)/len(df)*100:.1f}%)")
print(f"After community:  {len(df_after):,} records ({len(df_after)/len(df)*100:.1f}%)")
print(f"Total split:      {len(df_before) + len(df_after):,} (verification: {len(df_before) + len(df_after) == len(df)})")


# Remove Temporal Classification Columns

print("\nRemoving temporal classification columns...")

columns_to_drop = [
    'acc_cer_start_date',      # Community start date
    'has_acc_cer',              # Binary community indicator
    'months_since_acc_cer',     # Timeline relative to community start
    'new_acc', 'new_cer', 'new_total',  # New community counts
    'n_acc', 'n_cer', 'n_acc_cer',      # Community counts
    'control'                   # Control group flag
]

df_before = df_before.drop(columns=columns_to_drop, errors='ignore')

print(f"Remaining columns: {len(df_before.columns)}")
print(f"  Sample columns: {', '.join(df_before.columns[:8].tolist())}...")


# Save Final Datasets

print("\nSaving datasets...")

output_before = "4_SeparateData_output/data_before_1acc.csv"
output_after = "4_SeparateData_output/data_after_1acc.csv"

df_before.to_csv(output_before, index=False)
df_after.to_csv(output_after, index=False)

print(f" Saved: {output_before}")
print(f" Saved: {output_after}")

output_before = "../input/data_before_1acc.csv"
output_after = "../input/data_after_1acc.csv"

df_before.to_csv(output_before, index=False)
df_after.to_csv(output_after, index=False)

print(f" Saved: {output_before}")
print(f" Saved: {output_after}")


# Data Quality Summary

print("\nDATA QUALITY SUMMARY")

print("\nBefore Community:")
print(df_before.describe().T[['count', 'mean', 'min', 'max']])

print("\nAfter Community:")
print(df_after.describe().T[['count', 'mean', 'min', 'max']])

Loading integrated dataset...
Total records: 10,271
Date range: 2020-11-01 to 2025-10-01
Parishes: 173

Splitting data by community timeline...
Before community: 7,851 records (76.4%)
After community:  2,420 records (23.6%)
Total split:      10,271 (verification: True)

Removing temporal classification columns...
Remaining columns: 19
  Sample columns: parish_code, parish_name, municipality_name, date, year, month, org_energy, energy_kwh...

Saving datasets...
 Saved: 4_SeparateData_output/data_before_1acc.csv
 Saved: 4_SeparateData_output/data_after_1acc.csv
 Saved: ../input/data_before_1acc.csv
 Saved: ../input/data_after_1acc.csv

DATA QUALITY SUMMARY

Before Community:
                   count          mean           min           max
parish_code       7851.0  8.823185e+04  10402.000000  1.318110e+05
year              7851.0  2.022238e+03   2020.000000  2.024000e+03
month             7851.0  6.403388e+00      1.000000  1.200000e+01
org_energy        7851.0  3.653008e+06  15115.5230